# Corpus Preprocessing Pipeline
## Brazilian Chamber of Deputies Speech Data

---

This notebook implements a comprehensive, multi-level text preprocessing pipeline for parliamentary speech data. It produces multiple text representations optimized for different downstream analyses.

### Pipeline Overview

```
Raw Transcripts (transcricao)
        │
        ▼
┌─────────────────────────────────────────────────────────────────┐
│ STAGE 1: STRUCTURAL CLEANING                                    │
│   • Remove attachments and boilerplate                          │
│   • Extract primary speaker text                                │
│   • Quality filtering (length, duplicates)                      │
└─────────────────────────────────────────────────────────────────┘
        │
        ▼
    text_level_1 (Clean raw text)
        │
        ▼
┌─────────────────────────────────────────────────────────────────┐
│ STAGE 2: NORMALIZATION                                          │
│   • Lowercase                                                   │
│   • Remove punctuation/numbers                                  │
│   • Remove high-frequency irrelevant phrases                    │
└─────────────────────────────────────────────────────────────────┘
        │
        ▼
    text_level_2 (Normalized - for EMBEDDINGS)
        │
        ▼
┌─────────────────────────────────────────────────────────────────┐
│ STAGE 3: NAMED ENTITY REMOVAL (NER)                             │
│   • Remove person names (PER)                                   │
│   • Remove locations (LOC)                                      │
│   • Remove organizations (ORG)                                  │
│   ⚠️ MUST happen BEFORE stemming!                               │
└─────────────────────────────────────────────────────────────────┘
        │
        ▼
    text_level_3 (NER-cleaned - for VOCABULARY ANALYSIS)
        │
        ▼
┌─────────────────────────────────────────────────────────────────┐
│ STAGE 4: STEMMING & STOPWORD REMOVAL                            │
│   • Portuguese RSLP stemmer                                     │
│   • Remove stopwords                                            │
└─────────────────────────────────────────────────────────────────┘
        │
        ▼
    text_level_4 (Stemmed - for LDA/TF-IDF CLASSIFIER)
```

### Which Level to Use

| Level | Content | Best For |
|-------|---------|----------|
| 1 | Clean raw text | Qualitative reading, original quotes |
| 2 | Normalized (no punctuation) | **SBERT embeddings** (needs natural text) |
| 3 | NER removed | **Vocabulary analysis**, word frequencies |
| 4 | Stemmed + no stopwords | **LDA topic models**, TF-IDF classifier |

---

### Outputs

1. `data_corpus.parquet` - Full corpus with all text levels
2. `data_embeddings.parquet` - SBERT embeddings (from level_2)
3. `data_lda_topics.parquet` - LDA topic distributions (from level_4)

---

# Section 1: Setup & Configuration

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================

import os
import re
import warnings
from dataclasses import dataclass, field
from typing import List, Set, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

# NLP
import spacy
import nltk
from nltk.stem import RSLPStemmer
from nltk.corpus import stopwords

# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Embeddings
from sentence_transformers import SentenceTransformer

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

print("Imports complete.")

/Users/r2/Code/crawl-congresso/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports complete.


In [2]:
# =============================================================================
# NLTK DOWNLOADS
# =============================================================================

# Ensure NLTK resources are available
try:
    stopwords.words('portuguese')
except LookupError:
    print("Downloading NLTK stopwords...")
    nltk.download('stopwords')

try:
    nltk.data.find('stemmers/rslp')
except LookupError:
    print("Downloading NLTK RSLP stemmer...")
    nltk.download('rslp')

print("NLTK resources ready.")

NLTK resources ready.


In [3]:
# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class PreprocessingConfig:
    """All preprocessing parameters in one place."""
    
    # --- Paths ---
    input_path: str = '../data/raw/scrape_speeches.parquet'
    output_dir: str = '../data/processed/'
    
    # --- Quality Filtering ---
    min_text_length: int = 100  # Characters
    outlier_iqr_multiplier: float = 1.5  # For flagging unusually long speeches
    
    # --- Text Normalization ---
    min_word_length: int = 3  # Minimum characters per token
    idf_percentile_threshold: int = 10  # Remove words below this IDF percentile
    
    # --- NER ---
    spacy_model: str = 'pt_core_news_md'  # Portuguese medium model
    ner_entity_types: Set[str] = field(default_factory=lambda: {'PER', 'LOC', 'ORG'})
    ner_batch_size: int = 500
    
    # --- Embeddings ---
    sbert_model: str = 'paraphrase-multilingual-MiniLM-L12-v2'
    embedding_batch_size: int = 64
    
    # --- LDA ---
    lda_n_topics: int = 20
    lda_min_df: int = 300  # Minimum document frequency
    lda_max_df: float = 0.30  # Maximum document frequency
    lda_max_iter: int = 10
    
    # --- Parliamentary Stopwords ---
    parliamentary_stopwords: List[str] = field(default_factory=lambda: [
        'deputado', 'deputados', 'deputada', 'deputadas', 'parlamentar', 'parlamentares',
        'senhor', 'senhores', 'senhora', 'senhoras', 'vossa', 'excelencia', 'exa',
        'casa', 'plenario', 'mesa', 'projeto', 'projetos', 'lei', 'leis',
        'votacao', 'votacoes', 'sessao', 'sessoes', 'art', 'artigo', 'paragrafo',
        'hoje', 'aqui', 'agora', 'quero', 'dizer', 'fazer', 'vai', 'nesta', 'neste',
        'brasil', 'brasileiro', 'brasileiros', 'brasileira', 'brasileiras',
        'nacional', 'federal', 'uniao', 'estado', 'estados', 'governo',
        'presidente', 'ministro', 'ministros', 'camara', 'senado', 'congresso'
    ])

CFG = PreprocessingConfig()

# Create output directory
os.makedirs(CFG.output_dir, exist_ok=True)

print("Configuration:")
print(f"  Input: {CFG.input_path}")
print(f"  Output: {CFG.output_dir}")
print(f"  SpaCy model: {CFG.spacy_model}")
print(f"  SBERT model: {CFG.sbert_model}")
print(f"  LDA topics: {CFG.lda_n_topics}")

Configuration:
  Input: ../data/processed/scrape_speeches.parquet
  Output: ../data/processed/
  SpaCy model: pt_core_news_md
  SBERT model: paraphrase-multilingual-MiniLM-L12-v2
  LDA topics: 20


# Section 2: Data Loading

In [4]:
# =============================================================================
# LOAD RAW DATA
# =============================================================================

print("Loading raw data...")
df_raw = pd.read_parquet(CFG.input_path)

print(f"\nRaw data loaded:")
print(f"  Rows: {len(df_raw):,}")
print(f"  Columns: {df_raw.columns.tolist()}")

# Check for required column
if 'transcricao' not in df_raw.columns:
    raise ValueError("Required column 'transcricao' not found in data!")

print(f"\nSample transcription:")
print(df_raw['transcricao'].iloc[0][:500] if len(df_raw) > 0 else "No data")

Loading raw data...

Raw data loaded:
  Rows: 444,311
  Columns: ['dataHoraInicio', 'dataHoraFim', 'uriEvento', 'tipoDiscurso', 'urlTexto', 'urlAudio', 'urlVideo', 'keywords', 'sumario', 'transcricao', 'deputado_id', 'idLegislatura', 'faseEvento.titulo', 'faseEvento.dataHoraInicio', 'faseEvento.dataHoraFim']

Sample transcription:
O SR. ABEL MESQUITA JR. (DEM-RR. Pela ordem. Sem revisão do orador.) - Sr. Presidente, eu quero dizer para todos os colegas que o meu partido votou pela privatização dessas distribuidoras.
Mas eu também quero ressaltar aqui que fui eleito pelo Estado de Roraima, o único Estado da Federação que não é interligado com o Sistema Nacional de Energia.
	Agora há pouco, eu estava aqui ouvindo uma das pessoas mais experientes do Congresso Nacional dizer que quer salvar a União e as empresas. Eu, humild


In [5]:
# =============================================================================
# INITIALIZE REPORT DICTIONARY
# =============================================================================
# We track statistics at each step for the final preprocessing report

REPORT = {
    'initial_rows': len(df_raw),
    'initial_chars': df_raw['transcricao'].str.len().sum() if 'transcricao' in df_raw.columns else 0
}

# Working copy
df = df_raw.copy()

print(f"Initial corpus: {REPORT['initial_rows']:,} rows, {REPORT['initial_chars']:,} characters")

Initial corpus: 444,311 rows, 1,228,046,243.0 characters


# Section 3: Stage 1 - Structural Cleaning

This stage:
1. Removes NaN transcriptions
2. Removes attachments and boilerplate text
3. Extracts only the primary speaker's text
4. Filters by quality (length, duplicates)

In [6]:
# =============================================================================
# STRUCTURAL CLEANING FUNCTIONS
# =============================================================================

def remove_attachments(text: str, tipo_discurso: str = None) -> str:
    """
    Remove attachments and boilerplate from parliamentary transcripts.
    
    Brazilian parliamentary transcripts often contain:
    - "DISCURSO ENCAMINHADO" sections (submitted written speeches)
    - "A QUE SE REFERE" references to other documents
    - Various administrative annotations
    """
    if not isinstance(text, str):
        return ""
    
    # Handle submitted speeches
    if tipo_discurso == "DISCURSO ENCAMINHADO":
        text2 = re.sub(r"(?m)^.*ENCAMINHADO.*\n?", "", text)
        if not text2.strip():
            text2 = re.sub(r"(?m)^.*ENCAMINHADO[^a-z]*(?=[a-z])", "", text, flags=re.I)
        return text2.strip()
    
    # Normalize state abbreviation formatting
    text = re.sub(r"\s*-\s*(?=\b[A-Z]{2}\b)", "-", text)
    
    # Remove everything after "ENCAMINHADO"
    lines = text.splitlines(keepends=True)
    for i, line in enumerate(lines):
        if "ENCAMINHADO" in line:
            text = "".join(lines[:i])
            break
    
    # Remove everything after "A QUE SE REFERE"
    lines = text.splitlines(keepends=True)
    for i, line in enumerate(lines):
        if "A QUE SE REFERE" in line:
            text = "".join(lines[:i])
            break
    
    # Clean up whitespace
    text = re.sub(r'(?m)^\s+', '', text)
    
    return text.strip()


def extract_primary_speaker(text: str) -> str:
    """
    Extract only the primary speaker's text from a transcript.
    
    Parliamentary transcripts may contain multiple speakers (e.g., during debates).
    This function identifies the first speaker and extracts only their portions.
    
    Speaker markers: "O SR. FULANO -" or "A SRA. FULANA -"
    """
    if not isinstance(text, str):
        return ""
    
    def extract_name(match_text):
        """Extract speaker name from match."""
        cleaned = re.sub(r'^.*SR\.?', '', match_text, flags=re.IGNORECASE)
        cleaned = re.sub(r'\(.*?\)', '', cleaned)  # Remove party/state
        cleaned = re.sub(r'-$', '', cleaned)
        return cleaned.strip().upper()
    
    # Pattern for speaker markers
    pattern = r'^(?:O|A)\s+(?:SR\.?|SRA\.?).+?-\s*(?=\s|$)'
    matches = list(re.finditer(pattern, text, re.MULTILINE | re.IGNORECASE))
    
    if not matches:
        return text  # No speaker markers found, return as-is
    
    # Parse segments
    segments = []
    last_idx = 0
    for m in matches:
        segments.append((text[last_idx:m.start()], m.group()))
        last_idx = m.end()
    segments.append((text[last_idx:], None))
    
    # Group by speaker
    name_to_text = {}
    for i, (before, match) in enumerate(segments[:-1]):
        if match:
            name = extract_name(match)
            after = segments[i + 1][0]
            name_to_text.setdefault(name, []).append(after)
    
    # Return first speaker's text
    first_name = extract_name(matches[0].group())
    return "".join(name_to_text.get(first_name, [])).strip()


print("Structural cleaning functions defined.")

Structural cleaning functions defined.


In [7]:
# =============================================================================
# APPLY STRUCTURAL CLEANING
# =============================================================================

print("Stage 1: Structural Cleaning")
print("="*70)

# Step 1.1: Drop NaN transcriptions
df = df.dropna(subset=['transcricao'])
REPORT['rows_after_nan_drop'] = len(df)
print(f"After dropping NaN: {len(df):,} rows")

# Step 1.2: Remove attachments
print("\nRemoving attachments...")
df['text_level_1'] = df.progress_apply(
    lambda r: remove_attachments(r['transcricao'], r.get('tipoDiscurso')), 
    axis=1
)

chars_after_attachments = df['text_level_1'].str.len().sum()
REPORT['chars_removed_attachments'] = REPORT['initial_chars'] - chars_after_attachments
print(f"Characters removed: {REPORT['chars_removed_attachments']:,}")

# Step 1.3: Extract primary speaker
print("\nExtracting primary speaker text...")
df['text_level_1'] = df['text_level_1'].progress_apply(extract_primary_speaker)

chars_after_speaker = df['text_level_1'].str.len().sum()
REPORT['chars_removed_speaker'] = chars_after_attachments - chars_after_speaker
print(f"Characters removed: {REPORT['chars_removed_speaker']:,}")

Stage 1: Structural Cleaning
After dropping NaN: 444,280 rows

Removing attachments...


100%|██████████| 444280/444280 [00:31<00:00, 13942.29it/s]


Characters removed: 110,524,006.0

Extracting primary speaker text...


100%|██████████| 444280/444280 [00:11<00:00, 38433.53it/s]


Characters removed: 75,279,317


In [8]:
# =============================================================================
# QUALITY FILTERING
# =============================================================================

print("\nApplying quality filters...")

rows_before = len(df)

# Remove empty/whitespace-only
df = df.dropna(subset=['text_level_1'])
mask_empty = df['text_level_1'].str.strip().eq('')
REPORT['rows_dropped_empty'] = mask_empty.sum()
df = df[~mask_empty]

# Remove too short
mask_short = df['text_level_1'].str.len() < CFG.min_text_length
REPORT['rows_dropped_short'] = mask_short.sum()
df = df[~mask_short]

# Remove duplicates
rows_before_dedup = len(df)
df = df.drop_duplicates(subset=['text_level_1'])
REPORT['rows_dropped_duplicates'] = rows_before_dedup - len(df)

REPORT['rows_after_quality'] = len(df)

print(f"  Dropped empty: {REPORT['rows_dropped_empty']:,}")
print(f"  Dropped short (<{CFG.min_text_length} chars): {REPORT['rows_dropped_short']:,}")
print(f"  Dropped duplicates: {REPORT['rows_dropped_duplicates']:,}")
print(f"  Remaining: {len(df):,} rows")


Applying quality filters...
  Dropped empty: 155
  Dropped short (<100 chars): 1,275
  Dropped duplicates: 70,449
  Remaining: 372,401 rows


In [9]:
# =============================================================================
# FLAG OUTLIERS (but don't remove them)
# =============================================================================

print("\nFlagging outliers by length...")

lengths = df['text_level_1'].str.len()
q1 = lengths.quantile(0.25)
q3 = lengths.quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + CFG.outlier_iqr_multiplier * iqr

df['is_outlier'] = lengths > upper_bound

REPORT['outlier_stats'] = {
    'q1': q1,
    'q3': q3,
    'iqr': iqr,
    'upper_bound': upper_bound,
    'n_outliers': df['is_outlier'].sum()
}

print(f"  Q1: {q1:,.0f}, Q3: {q3:,.0f}, IQR: {iqr:,.0f}")
print(f"  Upper bound: {upper_bound:,.0f}")
print(f"  Outliers flagged: {df['is_outlier'].sum():,} ({100*df['is_outlier'].mean():.1f}%)")


Flagging outliers by length...
  Q1: 761, Q3: 3,064, IQR: 2,303
  Upper bound: 6,518
  Outliers flagged: 19,978 (5.4%)


# Section 4: Stage 2 - Normalization

Create `text_level_2`: normalized text suitable for embeddings.

- Lowercase
- Remove punctuation and numbers
- Remove high-frequency irrelevant phrases
- Keep natural word order (important for SBERT)

In [10]:
# =============================================================================
# FIND IRRELEVANT PHRASES (High-frequency, low-information)
# =============================================================================

print("Stage 2: Normalization")
print("="*70)
print("\nIdentifying high-frequency, low-IDF phrases...")

# Use TF-IDF to find phrases that appear everywhere (low IDF = low information)
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),
    min_df=5
)

# Fit on sample for speed
sample_texts = df['text_level_1'].sample(min(50000, len(df)), random_state=42).tolist()
vectorizer.fit(sample_texts)

# Find low-IDF terms
idf = vectorizer.idf_
idf_threshold = np.percentile(idf, CFG.idf_percentile_threshold)
low_idf_idx = np.where(idf <= idf_threshold)[0]
irrelevant_phrases = set(np.array(vectorizer.get_feature_names_out())[low_idf_idx])

REPORT['irrelevant_phrases_count'] = len(irrelevant_phrases)
REPORT['irrelevant_phrases_sample'] = list(irrelevant_phrases)[:15]

print(f"Found {len(irrelevant_phrases)} irrelevant phrases")
print(f"Sample: {REPORT['irrelevant_phrases_sample']}")

Stage 2: Normalization

Identifying high-frequency, low-IDF phrases...
Found 1001 irrelevant phrases
Sample: ['fundamental', 'discurso', 'geral', 'com os', 'la', 'quando', 'seria', 'tenha', 'texto', 'faz', 'muito', 'seguinte', 'cerca', 'brasileira', 'deputadas']


In [11]:
# =============================================================================
# CREATE TEXT_LEVEL_2 (Normalized)
# =============================================================================

def normalize_text(text: str, irrelevant: Set[str], min_word_len: int = 3) -> str:
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    # Keep only letters and spaces FIRST (before removing phrases)
    text = re.sub(r"[^a-záéíóúâêîôûãõç\s]", " ", text)
    
    # Split into words
    words = text.split()
    
    # Filter: remove irrelevant words AND short words
    words = [w for w in words if w not in irrelevant and len(w) >= min_word_len]
    
    return " ".join(words)


print("\nCreating text_level_2 (normalized)...")
df['text_level_2'] = df['text_level_1'].progress_apply(
    lambda x: normalize_text(x, irrelevant_phrases, CFG.min_word_length)
)

print(f"\nSample text_level_2:")
print(df['text_level_2'].iloc[0][:300] if len(df) > 0 else "No data")


Creating text_level_2 (normalized)...


100%|██████████| 372401/372401 [00:29<00:00, 12796.34it/s]



Sample text_level_2:
votou privatização distribuidoras ressaltar eleito roraima federação interligado energia ouvindo experientes salvar humildemente salvar roraima interligado votem privatização distribuidoras


# Section 5: Stage 3 - Named Entity Removal (NER)

Create `text_level_3`: text with named entities removed.

NER must be applied to `text_level_2` (normalized but NOT stemmed).

Why? SpaCy's NER model was trained on natural Portuguese text. If we stem first:
- "João Silva" becomes "joã silv"
- SpaCy can't recognize this as a person name
- The entity slips through and contaminates our vocabulary

In [15]:
# =============================================================================
# LOAD SPACY MODEL
# =============================================================================

print("Stage 3: Named Entity Removal")
print("="*70)
print(f"\nLoading SpaCy model: {CFG.spacy_model}...")

# Disable components we don't need - only keep tok2vec (required by NER) and ner
disable_components = ['morphologizer', 'parser', 'lemmatizer', 'attribute_ruler']

try:
    nlp = spacy.load(CFG.spacy_model, disable=disable_components)
    print(f"Model loaded successfully.")
    print(f"Active pipeline: {nlp.pipe_names}")
    print(f"Disabled (for speed): {disable_components}")
except OSError:
    print(f"WARNING: Model '{CFG.spacy_model}' not found!")
    print("Attempting to download...")
    os.system(f"python -m spacy download {CFG.spacy_model}")
    nlp = spacy.load(CFG.spacy_model, disable=disable_components)

Stage 3: Named Entity Removal

Loading SpaCy model: pt_core_news_md...
Model loaded successfully.
Active pipeline: ['tok2vec', 'ner']
Disabled (for speed): ['morphologizer', 'parser', 'lemmatizer', 'attribute_ruler']


In [16]:
# =============================================================================
# APPLY NER TO TEXT_LEVEL_2 (NOT stemmed!)
# =============================================================================

print(f"\nApplying NER to remove: {CFG.ner_entity_types}")
print("This may take a while for large corpora...")

# Process in batches with spacy.pipe for efficiency
texts = df['text_level_2'].tolist()

cleaned_texts = []
entity_counts = {'PER': 0, 'LOC': 0, 'ORG': 0, 'OTHER': 0}

for doc in tqdm(nlp.pipe(texts, batch_size=CFG.ner_batch_size, n_process=-1), 
                total=len(texts), desc="NER Processing"):
    # Keep tokens that are NOT named entities we want to remove
    tokens = []
    for token in doc:
        if token.ent_type_ in CFG.ner_entity_types:
            entity_counts[token.ent_type_] = entity_counts.get(token.ent_type_, 0) + 1
        else:
            tokens.append(token.text)
    cleaned_texts.append(" ".join(tokens))

df['text_level_3'] = cleaned_texts

REPORT['ner_entities_removed'] = entity_counts
print(f"\nEntities removed:")
for ent_type, count in entity_counts.items():
    if count > 0:
        print(f"  {ent_type}: {count:,}")


Applying NER to remove: {'ORG', 'LOC', 'PER'}
This may take a while for large corpora...


NER Processing: 100%|██████████| 372401/372401 [41:01<00:00, 151.30it/s]  



Entities removed:
  PER: 2,196,276
  LOC: 653,282
  ORG: 114,183


In [17]:
# =============================================================================
# VERIFY NER WORKED
# =============================================================================

print("\nVerification - Sample before and after NER:")
print("\nLevel 2 (with names):")
print(df['text_level_2'].iloc[0][:200])
print("\nLevel 3 (NER removed):")
print(df['text_level_3'].iloc[0][:200])

# checkpoint
output_path = os.path.join(CFG.output_dir, 'data_corpus.parquet')
df.to_parquet(output_path, compression='brotli', index=False)


Verification - Sample before and after NER:

Level 2 (with names):
votou privatização distribuidoras ressaltar eleito roraima federação interligado energia ouvindo experientes salvar humildemente salvar roraima interligado votem privatização distribuidoras

Level 3 (NER removed):
votou privatização distribuidoras ressaltar eleito federação interligado energia ouvindo experientes salvar humildemente salvar interligado votem privatização distribuidoras


# Section 6: Stage 4 - Stemming & Stopword Removal

Create `text_level_4`: stemmed text for topic modeling and classification.

Applied to `text_level_3` (NER-cleaned), so:
- No named entities to confuse the stemmer
- Clean vocabulary for LDA/TF-IDF

In [5]:
# =============================================================================
# PREPARE STEMMER AND STOPWORDS
# =============================================================================

print("Stage 4: Stemming & Stopword Removal")
print("="*70)

# Initialize stemmer
stemmer = RSLPStemmer()

# Combine NLTK stopwords with parliamentary stopwords
stop_words = set(stopwords.words('portuguese'))
stop_words.update(CFG.parliamentary_stopwords)

# Also add stemmed versions of stopwords
stop_words_stemmed = {stemmer.stem(w) for w in stop_words}
stop_words.update(stop_words_stemmed)

print(f"Total stopwords: {len(stop_words)}")
print(f"Sample: {list(stop_words)[:20]}")

Stage 4: Stemming & Stopword Removal
Total stopwords: 336
Sample: ['lhe', 'esses', 'houveremos', 'quando', 'estivér', 'sen', 'delas', 'plenari', 'artig', 'essas', 'e', 'eu', 'houvér', 'você', 'será', 'fazer', 'houvera', 'qu', 'tiver', 'excelenc']


In [19]:
# =============================================================================
# APPLY STEMMING TO TEXT_LEVEL_3
# =============================================================================

def stem_and_remove_stops(text: str, stemmer, stop_words: Set[str]) -> str:
    """
    Apply Portuguese RSLP stemming and remove stopwords.
    """
    if not isinstance(text, str):
        return ""
    
    tokens = text.split()
    stemmed = []
    
    for token in tokens:
        if token not in stop_words:
            stem = stemmer.stem(token)
            if stem not in stop_words and len(stem) >= 3:
                stemmed.append(stem)
    
    return " ".join(stemmed)


print("\nApplying stemming and stopword removal...")
df['text_level_4'] = df['text_level_3'].progress_apply(
    lambda x: stem_and_remove_stops(x, stemmer, stop_words)
)

print(f"\nSample text_level_4:")
print(df['text_level_4'].iloc[0][:200] if len(df) > 0 else "No data")


Applying stemming and stopword removal...


100%|██████████| 372401/372401 [08:54<00:00, 696.57it/s] 



Sample text_level_4:
vot priva distribu ressalt eleit interlig energ ouv experi salv humild salv interlig vot priva distribu


# Section 7: Save Corpus

In [20]:
# =============================================================================
# PREPARE FINAL DATAFRAME
# =============================================================================

print("Preparing final corpus...")

# Select columns to keep
metadata_cols = [col for col in df.columns 
                 if col not in ['transcricao', 'text_level_1', 'text_level_2', 
                                'text_level_3', 'text_level_4', 'is_outlier']]
text_cols = ['text_level_1', 'text_level_2', 'text_level_3', 'text_level_4', 'is_outlier']

final_cols = metadata_cols + text_cols
df_corpus = df[final_cols].copy()

print(f"\nFinal corpus shape: {df_corpus.shape}")
print(f"Columns: {df_corpus.columns.tolist()}")

Preparing final corpus...

Final corpus shape: (372401, 19)
Columns: ['dataHoraInicio', 'dataHoraFim', 'uriEvento', 'tipoDiscurso', 'urlTexto', 'urlAudio', 'urlVideo', 'keywords', 'sumario', 'deputado_id', 'idLegislatura', 'faseEvento.titulo', 'faseEvento.dataHoraInicio', 'faseEvento.dataHoraFim', 'text_level_1', 'text_level_2', 'text_level_3', 'text_level_4', 'is_outlier']


In [21]:
# =============================================================================
# SAVE CORPUS
# =============================================================================

output_path = os.path.join(CFG.output_dir, 'data_corpus.parquet')
df_corpus.to_parquet(output_path, compression='brotli', index=False)

print(f"\nCorpus saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1e6:.1f} MB")


Corpus saved to: ../data/processed/data_corpus.parquet
File size: 569.7 MB


# Section 8: Generate SBERT Embeddings

We use `text_level_2` (normalized but not stemmed) because:
- SBERT models expect natural language
- Stemming destroys semantic information
- NER removal is not needed (embeddings are robust to names)

In [22]:
# # =============================================================================
# # LOAD SBERT MODEL
# # =============================================================================

# print("Section 8: SBERT Embeddings")
# print("="*70)
# print(f"\nLoading SBERT model: {CFG.sbert_model}...")

# sbert_model = SentenceTransformer(CFG.sbert_model)
# print(f"Model loaded. Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")

In [23]:
# # =============================================================================
# # GENERATE EMBEDDINGS
# # =============================================================================

# print(f"\nGenerating embeddings for {len(df_corpus):,} speeches...")
# print("This will take a while. Consider using GPU if available.")

# texts = df_corpus['text_level_2'].fillna('').tolist()

# embeddings = sbert_model.encode(
#     texts,
#     batch_size=CFG.embedding_batch_size,
#     show_progress_bar=True,
#     convert_to_numpy=True
# )

# print(f"\nEmbeddings shape: {embeddings.shape}")

In [24]:
# # =============================================================================
# # SAVE EMBEDDINGS
# # =============================================================================

# # Create embeddings dataframe
# df_embeddings = df_corpus[['deputado_id', 'dataHoraInicio', 'text_level_2']].copy()
# df_embeddings['embedding'] = list(embeddings)

# # Save
# emb_path = os.path.join(CFG.output_dir, 'data_embeddings.parquet')
# df_embeddings.to_parquet(emb_path, compression='brotli', index=False)

# print(f"\nEmbeddings saved to: {emb_path}")
# print(f"File size: {os.path.getsize(emb_path) / 1e6:.1f} MB")

# Section 9: LDA Topic Modeling

We use `text_level_4` (stemmed, NER-cleaned) because:
- LDA works on word frequencies (bag-of-words)
- Stemming reduces vocabulary and groups related words
- NER removal prevents topics from clustering around specific names/places

In [6]:
# =============================================================================
# CREATE DOCUMENT-TERM MATRIX
# =============================================================================
df_corpus = pd.read_parquet(os.path.join(CFG.output_dir, 'data_corpus.parquet'))

print("Section 9: LDA Topic Modeling")
print("="*70)
print("\nCreating document-term matrix...")

# Use CountVectorizer (LDA needs raw counts, not TF-IDF)
count_vectorizer = CountVectorizer(
    min_df=CFG.lda_min_df,
    max_df=CFG.lda_max_df,
    ngram_range=(1, 2),
    stop_words=list(stop_words)
)

corpus_texts = df_corpus['text_level_4'].fillna('').tolist()
dtm = count_vectorizer.fit_transform(corpus_texts)

print(f"DTM shape: {dtm.shape} (documents × vocabulary)")
print(f"Vocabulary sample: {count_vectorizer.get_feature_names_out()[:20].tolist()}")

Section 9: LDA Topic Modeling

Creating document-term matrix...
DTM shape: (372401, 10182) (documents × vocabulary)
Vocabulary sample: ['abaf', 'abaix', 'abaix assin', 'abaix linh', 'abaix méd', 'abal', 'abaliz', 'abandon', 'abarc', 'abarrot', 'abast', 'abastec', 'abastec rural', 'abastec águ', 'abat', 'abc', 'abdic', 'abenço', 'aberr', 'abert']


In [7]:
# =============================================================================
# TRAIN LDA MODEL
# =============================================================================

print(f"\nTraining LDA model with {CFG.lda_n_topics} topics...")

lda_model = LatentDirichletAllocation(
    n_components=CFG.lda_n_topics,
    max_iter=CFG.lda_max_iter,
    learning_method='online',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lda_model.fit(dtm)
print("\nLDA model trained.")


Training LDA model with 20 topics...
iteration: 1 of max_iter: 10
iteration: 2 of max_iter: 10
iteration: 3 of max_iter: 10
iteration: 4 of max_iter: 10
iteration: 5 of max_iter: 10
iteration: 6 of max_iter: 10
iteration: 7 of max_iter: 10
iteration: 8 of max_iter: 10
iteration: 9 of max_iter: 10
iteration: 10 of max_iter: 10

LDA model trained.


In [8]:
# =============================================================================
# DISPLAY TOP WORDS PER TOPIC
# =============================================================================

print("\nTop 10 words per topic:")
print("="*70)

feature_names = count_vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda_model.components_):
    top_words_idx = topic.argsort()[:-11:-1]
    top_words = [feature_names[i] for i in top_words_idx]
    print(f"Topic {topic_idx:2d}: {', '.join(top_words)}")


Top 10 words per topic:
Topic  0: inform, solicit, investig, ped, denúnc, divulg, audi, public, decret, funcion
Topic  1: transport, águ, indígen, ambient, florest, rodov, estr, quilômetr, mat, port
Topic  2: indústr, export, vacin, comérci, import, exteri, comerc, americ, dól, unid
Topic  3: impost, fiscal, pag, tributár, receit, tribut, vet, gast, arrecad, financ
Topic  4: vot, entend, aprov, apresent, orient, retir, discut, encaminh, ped, lideranç
Topic  5: negr, comemor, cult, reconhec, viv, marc, histór, pernambuc, fund, dat
Topic  6: médic, mort, hospit, agent, doenç, crim, penal, trat, vítim, crimin
Topic  7: realiz, prefeit, municip, particip, event, visit, receb, local, vere, pesc
Topic  8: fal, acontec, viv, cheg, bolsonar, fic, pass, tir, perd, ouv
Topic  9: produt, rural, agricul, famili, aliment, pequen, agrícol, produz, agrár, assent
Topic 10: mulh, human, adolesc, igrej, crianç, sex, filh, deslig, microfon, pastor
Topic 11: pesquis, ger, produt, sustent, desenvolv, tecn

In [9]:
# =============================================================================
# COMPUTE TOPIC DISTRIBUTIONS
# =============================================================================

print("\nComputing topic distributions for all documents...")

topic_distributions = lda_model.transform(dtm)
dominant_topics = np.argmax(topic_distributions, axis=1)

df_corpus['topic_id'] = dominant_topics
df_corpus['topic_vector'] = list(topic_distributions)

print(f"Topic distributions shape: {topic_distributions.shape}")


Computing topic distributions for all documents...
Topic distributions shape: (372401, 20)


In [10]:
# =============================================================================
# SAVE LDA RESULTS
# =============================================================================

# Save topic distributions
df_lda = df_corpus[['deputado_id', 'dataHoraInicio', 'topic_id', 'topic_vector']].copy()
lda_path = os.path.join(CFG.output_dir, 'data_lda_topics.parquet')
df_lda.to_parquet(lda_path, compression='brotli', index=False)

print(f"\nLDA results saved to: {lda_path}")

# Save DTM for later use
import pickle
dtm_path = os.path.join(CFG.output_dir, 'dtm_level_4.pkl')
with open(dtm_path, 'wb') as f:
    pickle.dump({'dtm': dtm, 'vectorizer': count_vectorizer, 'lda_model': lda_model}, f)
print(f"DTM and models saved to: {dtm_path}")


LDA results saved to: ../data/processed/data_lda_topics.parquet
DTM and models saved to: ../data/processed/dtm_level_4.pkl


# Section 10: Preprocessing Report

In [ ]:
# =============================================================================
# FINAL PREPROCESSING REPORT
# =============================================================================

print("\n")
print("="*80)
print("PREPROCESSING REPORT")
print("="*80)

print("\n1. INITIAL STATE")
print(f"   Initial rows: {REPORT['initial_rows']:,}")
print(f"   Initial characters: {REPORT['initial_chars']:,}")

print("\n2. STRUCTURAL CLEANING")
print(f"   Rows after NaN drop: {REPORT['rows_after_nan_drop']:,}")
print(f"   Characters removed (attachments): {REPORT['chars_removed_attachments']:,}")
print(f"   Characters removed (speaker extraction): {REPORT['chars_removed_speaker']:,}")

print("\n3. QUALITY FILTERING")
print(f"   Dropped (empty): {REPORT['rows_dropped_empty']:,}")
print(f"   Dropped (short): {REPORT['rows_dropped_short']:,}")
print(f"   Dropped (duplicates): {REPORT['rows_dropped_duplicates']:,}")
print(f"   Final rows: {REPORT['rows_after_quality']:,}")

print("\n4. OUTLIER DETECTION")
stats = REPORT['outlier_stats']
print(f"   Q1 length: {stats['q1']:,.0f}")
print(f"   Q3 length: {stats['q3']:,.0f}")
print(f"   Upper bound: {stats['upper_bound']:,.0f}")
print(f"   Outliers flagged: {stats['n_outliers']:,}")

print("\n5. NORMALIZATION")
print(f"   Irrelevant phrases removed: {REPORT['irrelevant_phrases_count']}")
print(f"   Sample: {REPORT['irrelevant_phrases_sample'][:5]}")

print("\n6. NAMED ENTITY REMOVAL")
for ent, count in REPORT['ner_entities_removed'].items():
    if count > 0:
        print(f"   {ent}: {count:,}")

print("\n7. OUTPUT FILES")
print(f"   Corpus: {CFG.output_dir}data_corpus.parquet")
print(f"   Embeddings: {CFG.output_dir}data_embeddings.parquet")
print(f"   LDA Topics: {CFG.output_dir}data_lda_topics.parquet")

print("\n" + "="*80)
print("PREPROCESSING COMPLETE")
print("="*80)